# TRAIN

In [ ]:
# ========================== TRAIN PHASE ==========================
from google.colab import drive
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, normalize
import joblib
from pathlib import Path
import re

# Mount Drive
drive.mount('/content/drive')

# Load dữ liệu
file_path = '/content/drive/MyDrive/datasets/data_train.csv'
df = pd.read_csv(file_path)

print(f"Tổng records: {len(df)}")
print(f"Số bài hát duy nhất: {df['track_id'].nunique()}")

# Tạo catalog bài hát duy nhất
tracks_df = df.drop_duplicates(subset='track_id').copy()
tracks_df = tracks_df.reset_index(drop=True)

print(f"Số bài hát ban đầu: {len(tracks_df)}")

# Clean lyrics
print("\nĐang clean lyrics...")
def clean_lyrics(text):
    if pd.isna(text) or text == '':
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

tracks_df['lyrics_clean'] = tracks_df['lyrics'].apply(clean_lyrics)

# Tạo cột has_lyrics
print("\nĐang tạo cột has_lyrics...")
tracks_df['has_lyrics'] = (
    (tracks_df['lyrics_clean'] != '') &
    (tracks_df['lyrics_clean'].str.len() >= 20)
)

print(f"Số bài có lyrics hợp lệ: {tracks_df['has_lyrics'].sum()}")
print(f"Số bài không có lyrics: {len(tracks_df) - tracks_df['has_lyrics'].sum()}")

# Giữ toàn bộ bài hát (không filter)
tracks_df = tracks_df.dropna(subset=['track_id', 'title', 'artist_name']).reset_index(drop=True)
print(f"Số bài hát cuối cùng trong catalog: {len(tracks_df)}")

# Xây dựng feature matrices
print("\nĐang xử lý TF-IDF lyrics...")
tfidf_lyrics = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2), min_df=2)
lyrics_tfidf = tfidf_lyrics.fit_transform(tracks_df['lyrics_clean'])
lyrics_matrix = normalize(lyrics_tfidf)

print("\nĐang xử lý audio features...")
audio_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_s']
audio_data = tracks_df[audio_cols].fillna(0)
scaler_audio = StandardScaler()
audio_scaled = scaler_audio.fit_transform(audio_data)
audio_matrix = normalize(audio_scaled)
audio_matrix = np.array(audio_matrix)

print("\nĐang xử lý metadata...")
tracks_df['metadata_text'] = (
    tracks_df['artist_name'].str.lower().fillna('') + ' ' +
    tracks_df['artist_clean'].str.lower().fillna('') + ' ' +
    tracks_df['main_artist'].str.lower().fillna('') + ' ' +
    tracks_df['decade'].astype(str) + ' ' +
    tracks_df['year'].astype(str) + ' ' +
    tracks_df['explicit'].astype(str) + ' ' +
    tracks_df['popularity'].astype(str)
)
tfidf_metadata = TfidfVectorizer(max_features=5000, stop_words='english')
metadata_tfidf = tfidf_metadata.fit_transform(tracks_df['metadata_text'])
metadata_matrix = normalize(metadata_tfidf)

# Precompute similarity matrices
print("\nĐang tính similarity matrices...")
sim_lyrics = lyrics_matrix.dot(lyrics_matrix.T)
if sp.issparse(sim_lyrics): sim_lyrics = sim_lyrics.toarray()

sim_audio = audio_matrix.dot(audio_matrix.T)
sim_metadata = metadata_matrix.dot(metadata_matrix.T)
if sp.issparse(sim_metadata): sim_metadata = sim_metadata.toarray()

print("Precompute hoàn tất!")

# Lưu tất cả
save_path = '/content/drive/MyDrive/late_fusion/'
Path(save_path).mkdir(parents=True, exist_ok=True)

catalog_to_save = tracks_df[['track_id', 'title', 'artist_name', 'year', 'popularity', 'decade', 'has_lyrics']]
catalog_to_save.to_csv(save_path + 'tracks_catalog.csv', index=False)

joblib.dump(tfidf_lyrics, save_path + 'tfidf_lyrics.pkl')
joblib.dump(scaler_audio, save_path + 'scaler_audio.pkl')
joblib.dump(tfidf_metadata, save_path + 'tfidf_metadata.pkl')

np.savez_compressed(save_path + 'similarity_matrices.npz',
                    sim_lyrics=sim_lyrics, sim_audio=sim_audio, sim_metadata=sim_metadata)

print("\n=== TRAIN PHASE HOÀN TẤT ===")
print(f"Catalog lưu tại: {save_path}tracks_catalog.csv")
print(f"Tổng bài hát: {len(tracks_df)}, có lyrics: {tracks_df['has_lyrics'].sum()}")

Mounted at /content/drive
Tổng records: 1516735
Số bài hát duy nhất: 11256
Số bài hát ban đầu: 11256

Đang clean lyrics...

Đang tạo cột has_lyrics...
Số bài có lyrics hợp lệ: 6725
Số bài không có lyrics: 4531
Số bài hát cuối cùng trong catalog: 11256

Đang xử lý TF-IDF lyrics...

Đang xử lý audio features...

Đang xử lý metadata...

Đang tính similarity matrices...
Precompute hoàn tất!

=== TRAIN PHASE HOÀN TẤT ===
Catalog lưu tại: /content/drive/MyDrive/late_fusion/tracks_catalog.csv
Tổng bài hát: 11256, có lyrics: 6725


# TEST

In [ ]:
class LateFusionRecommender:
    def __init__(
        self,
        sim_lyrics: np.ndarray,
        sim_audio: np.ndarray,
        sim_metadata: np.ndarray,
        catalog: pd.DataFrame,
        default_weights_has_lyrics: dict = None,
        default_weights_no_lyrics: dict = None
    ):
        self.sim_lyrics = sim_lyrics
        self.sim_audio = sim_audio
        self.sim_metadata = sim_metadata
        self.catalog = catalog.reset_index(drop=True)
        self.N = len(catalog)

        self.default_weights_has_lyrics = default_weights_has_lyrics or {'lyrics': 0.4, 'audio': 0.4, 'metadata': 0.2}
        self.default_weights_no_lyrics = default_weights_no_lyrics or {'audio': 0.5, 'metadata': 0.5}

        self.track_to_idx = {tid: idx for idx, tid in enumerate(catalog['track_id'])}

        print(f"Recommender sẵn sàng với {self.N} bài hát")
        print(f"   → Có lyrics: {catalog['has_lyrics'].sum()} bài")

    def _get_adaptive_weights(self, seed_track_id: str, weights: dict = None):
        if seed_track_id not in self.track_to_idx:
            raise ValueError(f"Track ID '{seed_track_id}' không có trong catalog!")

        idx = self.track_to_idx[seed_track_id]
        has_lyrics = self.catalog.loc[idx, 'has_lyrics'] if 'has_lyrics' in self.catalog.columns else True

        if weights is not None:
            w = {'lyrics': 0.0, 'audio': 0.0, 'metadata': 0.0}
            w.update(weights)
            if not has_lyrics and w['lyrics'] > 0:
                print(f"Cảnh báo: Seed không có lyrics → bỏ lyrics weight")
                w['lyrics'] = 0.0
            total = sum(w.values())
            if total > 0:
                for k in w: w[k] /= total
            return w, has_lyrics
        else:
            if has_lyrics:
                return self.default_weights_has_lyrics.copy(), True
            else:
                return self.default_weights_no_lyrics.copy(), False

    def recommend(self, seed_track_id: str, top_k: int = 10, weights: dict = None) -> pd.DataFrame:
        idx = self.track_to_idx[seed_track_id]

        scores_lyrics = self.sim_lyrics[idx]
        scores_audio = self.sim_audio[idx]
        scores_metadata = self.sim_metadata[idx]

        w, has_lyrics = self._get_adaptive_weights(seed_track_id, weights)

        fused = np.zeros(self.N)
        if w.get('lyrics', 0) > 0:
            fused += w['lyrics'] * scores_lyrics
        fused += w['audio'] * scores_audio
        fused += w['metadata'] * scores_metadata

        fused[idx] = -np.inf
        top_indices = np.argsort(fused)[::-1][:top_k]

        recs = self.catalog.iloc[top_indices].copy()
        recs['score'] = fused[top_indices]
        recs = recs.reset_index(drop=True)

        recs.attrs['used_weights'] = w
        recs.attrs['seed_has_lyrics'] = has_lyrics
        recs.attrs['seed_track_id'] = seed_track_id

        return recs

    def print_seed_info(self, seed_track_id: str):
        if seed_track_id not in self.track_to_idx:
            print("Track không tồn tại!")
            return
        row = self.catalog.iloc[self.track_to_idx[seed_track_id]]
        print(f"\nSeed: {row['title']} - {row['artist_name']}")
        print(f"Has lyrics: {'Có' if row.get('has_lyrics', True) else 'Không'}")

In [ ]:
# ========================== TEST RECOMMENDER ==========================
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/late_fusion/'

# -------------------------- LOAD MEMORY-MAPPED -------------------------
print("Đang load similarity matrices dưới dạng memory-mapped (tiết kiệm RAM)...")

data = np.load(save_path + 'similarity_matrices.npz', mmap_mode='r')
# mmap_mode='r' → chỉ đọc từ disk khi cần, không load hết vào RAM

sim_lyrics = data['sim_lyrics']
sim_audio = data['sim_audio']
sim_metadata = data['sim_metadata']

catalog = pd.read_csv(save_path + 'tracks_catalog.csv')
catalog = catalog.reset_index(drop=True)

print(f"Loaded catalog: {len(catalog)} bài hát")
print(f"Số bài có lyrics: {catalog['has_lyrics'].sum()}")

# -------------------------- Khởi tạo recommender --------------------------
recommender = LateFusionRecommender(
    sim_lyrics=sim_lyrics,
    sim_audio=sim_audio,
    sim_metadata=sim_metadata,
    catalog=catalog
)

# -------------------------- Test với bài CÓ lyrics --------------------------
print("\n" + "="*60)
print("TEST 1: Bài hát CÓ LYRICS")
print("="*60)

track_with_lyrics = catalog[catalog['has_lyrics'] == True]['track_id'].iloc[0]

recommender.print_seed_info(track_with_lyrics)

recs1 = recommender.recommend(track_with_lyrics, top_k=10)
print("\nTop 10 recommendations:")
print(recs1[['title', 'artist_name', 'score']].to_string(index=False))

# -------------------------- Test với bài KHÔNG lyrics --------------------------
no_lyrics_df = catalog[catalog['has_lyrics'] == False]

if not no_lyrics_df.empty:
    print("\n" + "="*60)
    print("TEST 2: Bài hát KHÔNG CÓ LYRICS")
    print("="*60)

    track_no_lyrics = no_lyrics_df['track_id'].iloc[0]

    recommender.print_seed_info(track_no_lyrics)

    recs2 = recommender.recommend(track_no_lyrics, top_k=10)
    print("\nTop 10 recommendations (chỉ dùng audio + metadata):")
    print(recs2[['title', 'artist_name', 'score']].to_string(index=False))
else:
    print("\nKhông có bài hát nào không có lyrics trong catalog để test.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang load similarity matrices dưới dạng memory-mapped (tiết kiệm RAM)...
Loaded catalog: 11256 bài hát
Số bài có lyrics: 6725
Recommender sẵn sàng với 11256 bài hát
   → Có lyrics: 6725 bài

TEST 1: Bài hát CÓ LYRICS

Seed: Don't Cry (Original) - Guns N' Roses
Has lyrics: Có

Top 10 recommendations:
                          title       artist_name    score
                        So Fine     Guns N' Roses 0.519929
Love Is On The Way (LP Version)       Saigon Kick 0.474201
           Hold On (LP Version)     Jamie Walters 0.446716
                      Not Sorry   The Cranberries 0.422639
                      Stargazer  Mother Love Bone 0.414585
                    This I Love     Guns N' Roses 0.411305
 I Won't See You Tonight Part 1 Avenged Sevenfold 0.410971
                      Breakdown     Guns N' Roses 0.408860
               Live And Let Die     Gun

# EVALUATE

In [ ]:
class RecommendationEvaluator:
    def evaluate(self, test_data: dict, recommendations: dict, k_values=[5, 10, 20]):
        results = {k: {'tp': 0, 'fp': 0, 'fn': 0, 'hits': 0} for k in k_values}
        total_users = len(test_data)

        for user_id, gt_tracks in test_data.items():
            if user_id not in recommendations:
                continue
            rec_list = recommendations[user_id]
            gt_set = set(gt_tracks)

            for k in k_values:
                top_k = set(rec_list[:k])
                hits = len(top_k & gt_set)

                results[k]['tp'] += hits
                results[k]['fp'] += len(top_k - gt_set)
                results[k]['fn'] += len(gt_set - top_k)
                if hits > 0:
                    results[k]['hits'] += 1

        summary = {'Num_Test_Users': total_users}
        for k in k_values:
            r = results[k]
            precision = r['tp'] / (r['tp'] + r['fp']) if (r['tp'] + r['fp']) > 0 else 0
            recall = r['tp'] / (r['tp'] + r['fn']) if (r['tp'] + r['fn']) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            hit_rate = r['hits'] / total_users

            summary[f'Precision@{k}'] = precision
            summary[f'Recall@{k}']    = recall
            summary[f'F1@{k}']        = f1
            summary[f'HitRate@{k}']  = hit_rate

        return summary

    def print_summary(self, summary):
        print("\n" + "="*60)
        print("           KẾT QUẢ ĐÁNH GIÁ HỆ THỐNG")
        print("="*60)
        print(f"Số user test: {summary['Num_Test_Users']}")
        print("")
        for k in [5, 10, 20]:
            print(f"@{k:2d} | Precision: {summary[f'Precision@{k}']:.4f} | "
                  f"Recall: {summary[f'Recall@{k}']:.4f} | "
                  f"F1: {summary[f'F1@{k}']:.4f} | "
                  f"HitRate: {summary[f'HitRate@{k}']:.4f}")
        print("="*60)

In [ ]:
# ========================== EVALUATION PHASE ==========================
import random

# Load data
df = pd.read_csv('/content/drive/MyDrive/datasets/data_test.csv')
user_tracks_dict = df.groupby('user_id')['track_id'].apply(list).to_dict()

# Lọc user có >=10 bài
valid_users = {u: t for u, t in user_tracks_dict.items() if len(t) >= 10}
print(f"Số user hợp lệ: {len(valid_users)}")

# Split 80/20
random.seed(42)
test_data = {}
train_seeds = {}
track_to_idx = {tid: idx for idx, tid in enumerate(catalog['track_id'])}

for user_id, tracks in valid_users.items():
    unique = list(set(tracks))
    random.shuffle(unique)
    split = int(0.8 * len(unique))
    train_seeds[user_id] = unique[:split]
    test_data[user_id] = unique[split:]

# Load recommender
recommender = LateFusionRecommender(
    sim_lyrics=data['sim_lyrics'],
    sim_audio=data['sim_audio'],
    sim_metadata=data['sim_metadata'],
    catalog=catalog
)

# Generate recs
recommendations = {}
for i, user_id in enumerate(test_data):
    rec_set = set()
    for seed in train_seeds[user_id]:
        if seed not in track_to_idx: continue
        try:
            recs = recommender.recommend(seed, top_k=20)
            rec_set.update(recs['track_id'].tolist())
        except: continue
    recommendations[user_id] = list(rec_set)[:20]

    if (i+1) % 200 == 0:
        print(f"Đã xử lý {i+1}/{len(test_data)} users")

# Đánh giá
evaluator = RecommendationEvaluator()
summary = evaluator.evaluate(test_data, recommendations, [5,10,20])
evaluator.print_summary(summary)

Số user hợp lệ: 17313
Recommender sẵn sàng với 11256 bài hát
   → Có lyrics: 6725 bài
Đã xử lý 200/17313 users
Đã xử lý 400/17313 users
Đã xử lý 600/17313 users
Đã xử lý 800/17313 users
Đã xử lý 1000/17313 users
Đã xử lý 1200/17313 users
Đã xử lý 1400/17313 users
Đã xử lý 1600/17313 users
Đã xử lý 1800/17313 users
Đã xử lý 2000/17313 users
Đã xử lý 2200/17313 users
Đã xử lý 2400/17313 users
Đã xử lý 2600/17313 users
Đã xử lý 2800/17313 users
Đã xử lý 3000/17313 users
Đã xử lý 3200/17313 users
Đã xử lý 3400/17313 users
Đã xử lý 3600/17313 users
Đã xử lý 3800/17313 users
Đã xử lý 4000/17313 users
Đã xử lý 4200/17313 users
Đã xử lý 4400/17313 users
Đã xử lý 4600/17313 users
Đã xử lý 4800/17313 users
Đã xử lý 5000/17313 users
Đã xử lý 5200/17313 users
Đã xử lý 5400/17313 users
Đã xử lý 5600/17313 users
Đã xử lý 5800/17313 users
Đã xử lý 6000/17313 users
Đã xử lý 6200/17313 users
Đã xử lý 6400/17313 users
Đã xử lý 6600/17313 users
Đã xử lý 6800/17313 users
Đã xử lý 7000/17313 users
Đã xử lý